# 03 — Live deployment, visual mode

Live camera inference on the Kria KV260 with **bounding boxes overlaid on a
video feed** and **interactive sliders** for tuning conf/IOU/etc. without
restarting the kernel.

For the **maximum-throughput** text-only variant (used for benchmarking and
thesis-grade FPS numbers), see `02_deploy_text.ipynb`.

The video preview costs ~3-5 ms per frame (JPEG encode + browser refresh),
so this notebook tops out around 40-50 fps end-to-end versus the text mode's
60 fps. The DPU inference rate is unchanged (~12 ms per frame); the
difference is purely the rendering path. Use this notebook for **demos and
qualitative tuning**, use the text notebook for **benchmark numbers**.

**Prerequisites**: see `02_deploy_text.ipynb` — identical.

**Stop the live loop**: stop button (■) in the JupyterLab toolbar.


## 1. Configuration

Edit `VARIANT` to pick the model, then run the rest top-to-bottom.

If launched via `scripts/kria/run_live.sh <variant>`, the script sets
`LPR_VARIANT` and `LPR_XMODEL` and these values are picked up from there
automatically.

`CONF_THRESH`, `IOU_THRESH`, and `MAX_DETECTIONS` are *initial* values; in
the visual notebook (`03_deploy_visual.ipynb`) you can adjust them live via
sliders without restarting the cell.


In [ ]:
import os

VARIANT        = os.environ.get("LPR_VARIANT", "yolov5n")
CONF_THRESH    = 0.30
IOU_THRESH     = 0.45
MAX_DETECTIONS = 300

# Status display refresh rates
STATUS_DT      = 1.00     # seconds between HTML status refreshes
DET_LOG_DT     = 0.40     # min seconds between detection event prints

print(f"VARIANT        = {VARIANT}")
print(f"CONF_THRESH    = {CONF_THRESH}")
print(f"IOU_THRESH     = {IOU_THRESH}")
print(f"MAX_DETECTIONS = {MAX_DETECTIONS}")


## 2. Suppress glog noise

The Vitis AI runtime prints `Check failed: r == 0 cannot set read range!` on
every model load. Harmless ("fingerprint-verification quirk") but floods the
notebook output. Setting these env vars before any `vart`/`xir` import keeps
only FATAL-level glog messages.


In [ ]:
import os
os.environ['GLOG_minloglevel']     = '3'
os.environ['GLOG_logtostderr']     = '0'
os.environ['GLOG_stderrthreshold'] = '3'


## 3. Imports + repo location

The repo root is normally `/home/ubuntu/KriaKv260_Model_Compiler` on the Kria.
We add it to `sys.path` so `lpr_pipeline.deploy.*` is importable.


In [ ]:
import sys
from pathlib import Path

candidates = [
    os.environ.get("REPO_ROOT"),
    "/home/ubuntu/KriaKv260_Model_Compiler",
    str(Path.cwd().parent),
    str(Path.cwd()),
]
for cand in candidates:
    if cand and (Path(cand) / "lpr_pipeline").is_dir():
        REPO_ROOT = cand
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break
else:
    raise RuntimeError(
        "Could not locate lpr_pipeline. Set REPO_ROOT env var or run via "
        "scripts/kria/run_live.sh which sets it for you."
    )

print(f"REPO_ROOT = {REPO_ROOT}")

from lpr_pipeline.shared.models import get_spec

spec = get_spec(VARIANT)
XMODEL = os.environ.get(
    "LPR_XMODEL",
    f"/home/ubuntu/xmodels_vai35/{VARIANT}/{VARIANT}_kv260.xmodel",
)

print(f"spec   = family={spec.family} imgsz={spec.imgsz} "
      f"nc={spec.nc} reg_max={spec.reg_max}")
print(f"xmodel = {XMODEL}")

if not Path(XMODEL).exists():
    raise FileNotFoundError(
        f"xmodel missing: {XMODEL}\n"
        f"From your laptop, sync it:\n"
        f"  bash scripts/host/03_sync_to_kria.sh ubuntu@<kria-ip> {VARIANT}"
    )

import time, threading
import numpy as np
import cv2
from IPython.display import display, HTML
from pynq_dpu import DpuOverlay

from lpr_pipeline.deploy import (
    ModelRunner, ThreadedCamera,
    draw_detections, draw_stats_overlay,
)

print(f"OpenCV {cv2.__version__}")


## 4. Load DPU overlay (program the FPGA)

The DPU bitstream is loaded once. After this cell runs, the FPGA fabric is
configured and we can hot-swap xmodels onto it via `overlay.load_model()`
without reprogramming the hardware.


In [ ]:
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded — FPGA programmed and ready for xmodels.")


## 5. Build the ModelRunner and warm it up

`ModelRunner` ties together the preprocessor (letterbox + normalize), the
DPU runner, and the decoder (`decode_yolov5u`). Warmup runs 5 inferences on
random input to settle JIT + caches.


In [ ]:
runner = ModelRunner(spec, XMODEL, overlay)

print(f"\nModelRunner built:")
print(f"  input  dims = {runner.input_dims}")
for i, d in enumerate(runner.output_dims):
    print(f"  output[{i}] = {d}")

print(f"\nWarming up (5 runs):")
warmup_times = runner.warmup(n=5, print_each=True)
print(f"\nSteady-state ≈ {min(warmup_times[2:]):.2f} ms (best of last 3)")


## 6. Pure inference benchmark (DPU + decode ceiling)

Measures maximum inference rate fed an in-memory frame — no camera, no
display. The reportable number is `mean_ms`; throughput is `1000 / mean_ms`.


In [ ]:
def benchmark_pure(n=200, warmup=20):
    # Pure-inference benchmark with per-stage breakdown.
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    for _ in range(warmup):
        runner.infer(frame)

    total_times = np.empty(n, dtype=np.float64)
    pre_times   = np.empty(n, dtype=np.float64)
    dpu_times   = np.empty(n, dtype=np.float64)
    dec_times   = np.empty(n, dtype=np.float64)

    for i in range(n):
        t0 = time.perf_counter()
        _, t = runner.infer(frame)
        total_times[i] = (time.perf_counter() - t0) * 1000
        pre_times[i]   = t["preprocess"]
        dpu_times[i]   = t["dpu"]
        dec_times[i]   = t["decode"]

    def stats(arr, label):
        print(f"  {label:>10s}  "
              f"mean={arr.mean():6.2f}  "
              f"p50={np.percentile(arr,50):6.2f}  "
              f"p95={np.percentile(arr,95):6.2f}  "
              f"p99={np.percentile(arr,99):6.2f}")

    print(f"=== {VARIANT} pure inference (n={n}) — timings in ms ===")
    stats(total_times, "total")
    stats(pre_times,   "preprocess")
    stats(dpu_times,   "dpu")
    stats(dec_times,   "decode")
    print(f"\n  → throughput = {1000 / total_times.mean():6.1f} fps  "
          f"(p95: {1000 / np.percentile(total_times, 95):.1f} fps)")
    return total_times, pre_times, dpu_times, dec_times

bench_total, bench_pre, bench_dpu, bench_dec = benchmark_pure()


## 7. Interactive controls

Sliders and toggles for live parameter tuning. Adjust these *while the live
loop runs in the next cell* — changes take effect on the next frame.


In [ ]:
import ipywidgets as widgets

# Sliders for the post-processing thresholds.
conf_slider = widgets.FloatSlider(
    value=CONF_THRESH, min=0.05, max=0.95, step=0.01,
    description="conf",
    style={"description_width": "80px"},
    continuous_update=True,
    layout=widgets.Layout(width="320px"),
)
iou_slider = widgets.FloatSlider(
    value=IOU_THRESH, min=0.05, max=0.95, step=0.01,
    description="IOU",
    style={"description_width": "80px"},
    continuous_update=True,
    layout=widgets.Layout(width="320px"),
)
max_det_slider = widgets.IntSlider(
    value=MAX_DETECTIONS, min=1, max=500, step=1,
    description="max det",
    style={"description_width": "80px"},
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)

# Toggles for rendering options.
labels_toggle = widgets.Checkbox(
    value=True, description="show labels",
    indent=False,
)
confidence_toggle = widgets.Checkbox(
    value=True, description="show confidence",
    indent=False,
)
stats_overlay_toggle = widgets.Checkbox(
    value=True, description="stats overlay on video",
    indent=False,
)

# Stop button — clicking it sets a flag the live loop checks.
stop_button = widgets.Button(
    description="■ Stop",
    button_style="danger",
    layout=widgets.Layout(width="120px"),
)
_stop_flag = {"stop": False}
def _on_stop_clicked(_):
    _stop_flag["stop"] = True
stop_button.on_click(_on_stop_clicked)

# Layout
controls = widgets.VBox([
    widgets.HBox([conf_slider, iou_slider]),
    widgets.HBox([max_det_slider, stop_button]),
    widgets.HBox([labels_toggle, confidence_toggle, stats_overlay_toggle]),
])
display(controls)

print("Sliders and toggles ready. They take effect once the live loop is running.")


## 8. Live loop — video preview with bounding boxes

Camera frames go through the runner and get drawn-on (boxes, labels, optional
stats overlay), then are JPEG-encoded and pushed to an `ipywidgets.Image`
widget. The text status panel above the video updates once per second as in
the text notebook.

**Stop**: click the **■ Stop** button in the controls cell above. (You can
also use Kernel → Interrupt, but the button is faster and doesn't disturb
the kernel.)

**FPS expectations**:
- DPU work: ~12 ms (unchanged from text notebook)
- JPEG encode + IPython widget update: ~5-8 ms additional
- End-to-end: 40-50 fps typical, vs 60 in the text mode

The DPU is idle 40% of the time waiting for the display path. For thesis
*throughput* claims, cite the text notebook's numbers; for *demo* purposes,
this notebook is the right tool.


In [ ]:
# Release prior camera handle if a previous run left one
try:
    cam.close()
    print("(closed prior camera)")
except (NameError, AttributeError):
    pass

cam = ThreadedCamera()
print(f"  camera: {cam.actual_width}x{cam.actual_height} "
      f"@ {cam.actual_fps:.0f} fps  BUFFERSIZE=4")

# Video output widget (JPEG-encoded frames written to .value)
video_widget = widgets.Image(format="jpeg", width=640, height=480)

# Text status above the video
status = display(
    HTML("<pre style='font-family:monospace'>starting...</pre>"),
    display_id=True,
)
display(video_widget)

# Reset stop flag from any prior run
_stop_flag["stop"] = False

# JPEG encode parameters — quality 80 is a good throughput/quality tradeoff.
jpeg_params = [int(cv2.IMWRITE_JPEG_QUALITY), 80]

# Class names for label drawing. For LPR (nc=1) this is just ["plate"].
class_names = ["plate"] if spec.nc == 1 else [f"class{i}" for i in range(spec.nc)]

n_inf, n_dets_total, n_frames_with_d = 0, 0, 0
unique_ids = set()
ema_pre, ema_dpu, ema_dec, ema_enc = 0.0, 0.0, 0.0, 0.0

t_start         = time.perf_counter()
last_status_t   = t_start

print(f"\n[ live {VARIANT}  •  use ■ Stop above to end ]\n")

try:
    while not _stop_flag["stop"]:
        frame, fid = cam.read_new()
        if frame is None:
            time.sleep(0.001)
            continue
        unique_ids.add(fid)

        # Pull current values from widgets
        conf  = conf_slider.value
        iou   = iou_slider.value
        max_d = max_det_slider.value
        show_lab  = labels_toggle.value
        show_conf = confidence_toggle.value
        show_stat = stats_overlay_toggle.value

        dets, t = runner.infer(frame, conf=conf, iou=iou, max_detections=max_d)
        n_inf += 1

        if ema_pre:
            ema_pre = 0.9 * ema_pre + 0.1 * t["preprocess"]
            ema_dpu = 0.9 * ema_dpu + 0.1 * t["dpu"]
            ema_dec = 0.9 * ema_dec + 0.1 * t["decode"]
        else:
            ema_pre, ema_dpu, ema_dec = t["preprocess"], t["dpu"], t["decode"]

        if dets:
            n_dets_total    += len(dets)
            n_frames_with_d += 1

        now = time.perf_counter()
        elapsed_so_far = now - t_start
        inf_fps_so_far = n_inf / elapsed_so_far if elapsed_so_far > 0 else 0
        ema_total = ema_pre + ema_dpu + ema_dec

        # Render: copy the frame (we don't own the camera buffer), draw boxes,
        # optional stats overlay, then JPEG encode.
        t_enc0 = time.perf_counter()
        annotated = frame.copy()
        draw_detections(
            annotated, dets,
            class_names=class_names,
            show_labels=show_lab,
            show_confidence=show_conf,
        )
        if show_stat:
            draw_stats_overlay(
                annotated,
                fps=inf_fps_so_far,
                inf_ms=ema_total,
                detections=len(dets),
            )
        ok, jpg = cv2.imencode(".jpg", annotated, jpeg_params)
        t_enc = (time.perf_counter() - t_enc0) * 1000

        if ema_enc:
            ema_enc = 0.9 * ema_enc + 0.1 * t_enc
        else:
            ema_enc = t_enc

        if ok:
            video_widget.value = jpg.tobytes()

        # Text status block (once per second)
        if (now - last_status_t) >= STATUS_DT:
            last_status_t = now
            elapsed = now - t_start
            inf_fps = n_inf / elapsed
            cam_fps = len(unique_ids) / elapsed
            hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
            html = (
                "<pre style='font-family:monospace;font-size:13px;"
                "background:#1e1e1e;color:#d4d4d4;padding:8px;"
                "border-radius:4px'>"
                f"<b style='color:#4ec9b0'>{VARIANT}</b>  "
                f"elapsed={elapsed:6.1f}s  frames={n_inf:6d}\n"
                f"<b style='color:#dcdcaa'>inf_fps</b>={inf_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>cam_fps</b>={cam_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>display_ms</b>={ema_enc:5.2f}\n"
                f"<b style='color:#9cdcfe'>pre</b>={ema_pre:4.2f}  "
                f"<b style='color:#9cdcfe'>dpu</b>={ema_dpu:5.2f}  "
                f"<b style='color:#9cdcfe'>dec</b>={ema_dec:4.2f}  "
                f"(<b style='color:#9cdcfe'>inf_total</b>={ema_total:5.2f} ms)\n"
                f"<b style='color:#c586c0'>detections</b>={n_dets_total}  "
                f"<b style='color:#c586c0'>hit_rate</b>={hit_pct:5.1f}%"
                "</pre>"
            )
            status.update(HTML(html))

except KeyboardInterrupt:
    print("\n[ stopped by Kernel→Interrupt ]")
finally:
    cam.close()
    elapsed = time.perf_counter() - t_start
    inf_fps = n_inf / elapsed if elapsed > 0 else 0
    cam_fps = len(unique_ids) / elapsed if elapsed > 0 else 0
    hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
    ema_total = ema_pre + ema_dpu + ema_dec

    if _stop_flag["stop"]:
        print("\n[ stopped by user via ■ Stop button ]")

    print(f"\n=== final stats — {VARIANT} (visual mode) ===")
    print(f"  total time            : {elapsed:8.2f} s")
    print(f"  frames inferred       : {n_inf:8d}")
    print(f"  unique camera frames  : {len(unique_ids):8d}")
    print(f"  inference fps         : {inf_fps:8.2f}")
    print(f"  camera fps            : {cam_fps:8.2f}")
    print(f"  avg preprocess (ms)   : {ema_pre:8.2f}")
    print(f"  avg dpu        (ms)   : {ema_dpu:8.2f}")
    print(f"  avg decode     (ms)   : {ema_dec:8.2f}")
    print(f"  avg inf total  (ms)   : {ema_total:8.2f}")
    print(f"  avg display    (ms)   : {ema_enc:8.2f}")
    print(f"  total detections      : {n_dets_total:8d}")
    print(f"  frames with object    : {n_frames_with_d:8d}")
    print(f"  hit rate              : {hit_pct:8.2f} %")


## 9. Notes

### Reading the numbers

| Number | What it means |
|---|---|
| `inf_fps` | End-to-end throughput, **including the display path** |
| `cam_fps` | Camera's unique-frame rate (60 if camera tuning succeeded) |
| `display_ms` | JPEG encode + widget update time per frame |
| `pre / dpu / dec` | Per-stage inference time (unchanged from text notebook) |
| `inf_total` | `pre + dpu + dec` — what the DPU pipeline needs alone |

If `display_ms > 5` and `inf_fps < cam_fps`, the display path is the
bottleneck. Acceptable for this notebook — for a thesis-grade FPS number,
use `02_deploy_text.ipynb`.

### Adjusting parameters live

The sliders update `conf_slider.value` etc. as you drag them. The live loop
re-reads these on every iteration, so changes take effect on the next frame
(< 20 ms). No kernel restart needed.

The **stats overlay** toggle controls the text drawn onto the video itself
(top-left corner: FPS, inference time, detection count). The HTML status
block above the video is independent and always visible.

### Why two notebooks?

| Use case | Notebook |
|---|---|
| Thesis benchmarks, "the model runs at X fps" | `02_deploy_text.ipynb` |
| Live demo, presentation, parameter exploration | `03_deploy_visual.ipynb` |
| Debugging a specific detection | `03_deploy_visual.ipynb` |

The display path costs ~5-8 ms/frame; isolating it in its own notebook means
the benchmark numbers reflect what the DPU pipeline can really do, not what
ipywidgets can render.

### Switching models

Same as the text notebook: edit `VARIANT` in cell 1, restart the kernel.
